# Task 1.3 — What the Paper Claims to Improve
**Paper:** *Scalable Training of Mixture Models via Coresets* (Feldman, Faulkner, Krause — NIPS 2011)

---

## Main Baseline Method

The primary baseline the paper compares against empirically is **uniform random subsampling** — selecting a subset C from D by drawing each point with equal probability 1/n, then running standard (or weighted) EM on that uniform sample. The paper also compares against running EM on the **full dataset D** as the performance ceiling.

The theoretical baseline is the method of **Arora and Kannan (2005)** [Reference 3 in the paper], which provides a PTAS for fitting GMMs with near-optimal log-likelihood — but only for the special case where all k Gaussians are *identical spheres*, effectively reducing the problem to k-means.

---

## Limitation of the Baseline

**Uniform sampling** has a fundamental flaw: it allocates sample budget proportionally to cluster *size*, not to cluster *importance for the likelihood*. If the data has one large cluster with 99% of points and one small but diagnostically important cluster with 1% of points, a uniform sample of size m will almost certainly miss the small cluster entirely unless m is Ω(n) — making it no cheaper than the full dataset. The paper demonstrates this formally with a two-Gaussian counterexample (Section 3): with weights w₁ = 1/√n and w₂ = 1 − 1/√n, uniform sampling requires Ω(√n) points just to include one point from the minority Gaussian. The log-likelihood of the model fitted on the uniform sample can be made arbitrarily worse than the model fitted on the full data by moving the two means far apart.

The theoretical baseline (Arora-Kannan) has the limitation of requiring all Gaussians to be identical spheres — a very strong restriction that rules out the general covariance case.

---

## How the Proposed Method Overcomes the Limitation

The coreset method replaces uniform sampling with **importance-weighted adaptive sampling**: each data point receives a sampling probability proportional to its "difficulty" — a combination of how densely packed its local neighbourhood is (the 5/|D_b| term) and how far it is from the rough clustering B (the dist(x, B)² term). Points in small, sparse, or distant clusters are sampled with higher probability, ensuring they are represented in C regardless of their cluster's size. The corrective weights γ(x') then debias this non-uniform selection so the weighted likelihood on C matches the full likelihood. The result is a coreset of size O(dk³/ε²) that is completely *independent of n* and works for general ε-semi-spherical (non-identical, non-spherical) Gaussians.

---

## Condition Under Which the Paper's Method Would Not Outperform Uniform Sampling

The coreset method's advantage over uniform sampling shrinks and can disappear in the following scenario: **when all k Gaussian components have nearly equal mixing weights and similar variances, and when the dataset is very large but well-balanced**.

Here is the reasoning: The coreset's edge comes from identifying and over-sampling under-represented clusters. If all clusters contribute roughly equally to the data (balanced weights w_i ≈ 1/k), uniform sampling will naturally include proportional representation from each cluster, so its liability (missing small clusters) never triggers. Meanwhile, the coreset construction has additional overhead — building B requires O(ndk) time, and the importance weight computation adds complexity. For a perfectly balanced, well-separated GMM with large n, uniform sampling of size m = O(k·log(1/δ)/ε²) already provides a good approximation by standard concentration bounds, and the coreset provides no statistical advantage while incurring extra construction cost.

This is partially consistent with the paper's own observations: in the MNIST experiment (Figure 3a), the gap between coreset and uniform sample narrows as the training set size grows beyond ~1000 points, because at large sizes even uniform sampling becomes representative. The coreset's relative advantage is most pronounced at *small subset sizes*, which is exactly when balanced data would also be well-served by uniform sampling.